In [71]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import uuid


In [100]:
root_dir = Path.cwd()
if not (root_dir / "data").exists():
    root_dir = root_dir.parent
if not (root_dir / "data").exists():
    raise FileNotFoundError("Could not locate project root containing the data directory.")

bronze_dir = root_dir / "data" / "bronze"
bronze_dir.mkdir(parents=True, exist_ok=True)
silver_dir = root_dir / "data" / "silver"

source_path = bronze_dir / "bronze_wc_matches.csv"
output_path = root_dir / "data" / "silver"



In [73]:
silver_wc = pd.read_csv(source_path)

In [10]:
silver_wc.shape

(900, 18)

In [11]:
silver_wc.head()

,year,country,city,stage,home_team,away_team,home_score,away_score,outcome,win_conditions,winning_team,losing_team,date,month,dayofweek,source_name,load_timestamp,run_id
0,1930,Uruguay,Montevideo,Group 1,France,Mexico,4,1,H,NaN,France,Mexico,1930-07-13,Jul,Sunday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
1,1930,Uruguay,Montevideo,Group 4,Belgium,United States,0,3,A,NaN,United States,Belgium,1930-07-13,Jul,Sunday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
2,1930,Uruguay,Montevideo,Group 2,Brazil,Yugoslavia,1,2,A,NaN,Yugoslavia,Brazil,1930-07-14,Jul,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
3,1930,Uruguay,Montevideo,Group 3,Peru,Romania,1,3,A,NaN,Romania,Peru,1930-07-14,Jul,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
4,1930,Uruguay,Montevideo,Group 1,Argentina,France,1,0,H,NaN,Argentina,France,1930-07-15,Jul,Tuesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe


In [13]:
silver_wc.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   year            900 non-null    int64
 1   country         900 non-null    str  
 2   city            900 non-null    str  
 3   stage           900 non-null    str  
 4   home_team       900 non-null    str  
 5   away_team       900 non-null    str  
 6   home_score      900 non-null    int64
 7   away_score      900 non-null    int64
 8   outcome         900 non-null    str  
 9   win_conditions  62 non-null     str  
 10  winning_team    731 non-null    str  
 11  losing_team     731 non-null    str  
 12  date            900 non-null    str  
 13  month           900 non-null    str  
 14  dayofweek       900 non-null    str  
 15  source_name     900 non-null    str  
 16  load_timestamp  900 non-null    str  
 17  run_id          900 non-null    str  
dtypes: int64(3), str(15)
memory usage: 126.7 

In [15]:
silver_wc.isna().sum()

year                0
country             0
city                0
stage               0
home_team           0
away_team           0
home_score          0
away_score          0
outcome             0
win_conditions    838
winning_team      169
losing_team       169
date                0
month               0
dayofweek           0
source_name         0
load_timestamp      0
run_id              0
dtype: int64

In [32]:
home_teams = sorted(silver_wc["home_team"].dropna().unique())

for team in home_teams:
    print(team)

Algeria
Angola
Argentina
Australia
Austria
Belgium
Bolivia
Bosnia and Herzegovina
Brazil
Bulgaria
Cameroon
Canada
Chile
China PR
Colombia
Costa Rica
Croatia
Cuba
Czech Republic
Czechoslovakia
Denmark
East Germany
Ecuador
Egypt
El Salvador
England
FR Yugoslavia
France
Germany
Ghana
Greece
Haiti
Honduras
Hungary
Iceland
Iran
Iraq
Israel
Italy
Ivory Coast
Jamaica
Japan
Mexico
Morocco
Netherlands
New Zealand
Nigeria
North Korea
Northern Ireland
Norway
Panama
Paraguay
Peru
Poland
Portugal
Republic of Ireland
Romania
Russia
Saudi Arabia
Scotland
Senegal
Serbia
Slovakia
Slovenia
South Africa
South Korea
Soviet Union
Spain
Sweden
Switzerland
Togo
Trinidad and Tobago
Tunisia
Turkey
Ukraine
United Arab Emirates
United States
Uruguay
West Germany
Yugoslavia
Zaire


In [29]:
silver_wc.duplicated().sum()

np.int64(0)

In [20]:
print(silver_wc["winning_team"].isna())

0      False
1      False
2      False
3      False
4      False
       ...  
895    False
896    False
897    False
898    False
899    False
Name: winning_team, Length: 900, dtype: bool


In [33]:
silver_wc_clean = silver_wc.copy()

In [ ]:
silver_wc_clean.columns = (
    silver_wc_clean.columns
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [39]:
#convert strings
silver_wc_clean = silver_wc_clean.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

In [40]:
silver_wc_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   year            900 non-null    int64
 1   country         900 non-null    str  
 2   city            900 non-null    str  
 3   stage           900 non-null    str  
 4   home_team       900 non-null    str  
 5   away_team       900 non-null    str  
 6   home_score      900 non-null    int64
 7   away_score      900 non-null    int64
 8   outcome         900 non-null    str  
 9   win_conditions  62 non-null     str  
 10  winning_team    731 non-null    str  
 11  losing_team     731 non-null    str  
 12  date            900 non-null    str  
 13  month           900 non-null    str  
 14  dayofweek       900 non-null    str  
 15  source_name     900 non-null    str  
 16  load_timestamp  900 non-null    str  
 17  run_id          900 non-null    str  
dtypes: int64(3), str(15)
memory usage: 126.7 

In [41]:
silver_wc_clean

,year,country,city,stage,home_team,away_team,home_score,away_score,outcome,win_conditions,winning_team,losing_team,date,month,dayofweek,source_name,load_timestamp,run_id
0,1930,Uruguay,Montevideo,Group 1,France,Mexico,4,1,H,NaN,France,Mexico,1930-07-13,Jul,Sunday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
1,1930,Uruguay,Montevideo,Group 4,Belgium,United States,0,3,A,NaN,United States,Belgium,1930-07-13,Jul,Sunday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
2,1930,Uruguay,Montevideo,Group 2,Brazil,Yugoslavia,1,2,A,NaN,Yugoslavia,Brazil,1930-07-14,Jul,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
3,1930,Uruguay,Montevideo,Group 3,Peru,Romania,1,3,A,NaN,Romania,Peru,1930-07-14,Jul,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
4,1930,Uruguay,Montevideo,Group 1,Argentina,France,1,0,H,NaN,Argentina,France,1930-07-15,Jul,Tuesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,2018,Russia,Sochi,Quarterfinals,Russia,Croatia,2,2,A,Croatia won in penalties (4 - 3),Croatia,Russia,2018-07-07,Jul,Saturday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
896,2018,Russia,Saint Petersburg,Semifinals,France,Belgium,1,0,H,NaN,France,Belgium,2018-07-10,Jul,Tuesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
897,2018,Russia,Moscow,Semifinals,Croatia,England,2,1,H,Croatia won in AET,Croatia,England,2018-07-11,Jul,Wednesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
898,2018,Russia,Saint Petersburg,Third place,Belgium,England,2,0,H,NaN,Belgium,England,2018-07-14,Jul,Saturday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe


In [42]:
silver_wc_clean["load_timestamp_clean"] = (
    pd.to_datetime(silver_wc_clean["load_timestamp"], errors="coerce")
)

In [43]:
silver_wc_clean["date_clean"] = (
    pd.to_datetime(silver_wc_clean["date"], errors="coerce")
)

In [47]:
silver_wc_clean["year"] = silver_wc_clean["date_clean"].dt.year

In [48]:
silver_wc_clean.head()

,year,country,city,stage,home_team,away_team,home_score,away_score,outcome,win_conditions,winning_team,losing_team,date,month,dayofweek,source_name,load_timestamp,run_id,load_timestamp_clean,date_clean
0,1930,Uruguay,Montevideo,Group 1,France,Mexico,4,1,H,NaN,France,Mexico,1930-07-13,Jul,Sunday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1930-07-13
1,1930,Uruguay,Montevideo,Group 4,Belgium,United States,0,3,A,NaN,United States,Belgium,1930-07-13,Jul,Sunday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1930-07-13
2,1930,Uruguay,Montevideo,Group 2,Brazil,Yugoslavia,1,2,A,NaN,Yugoslavia,Brazil,1930-07-14,Jul,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1930-07-14
3,1930,Uruguay,Montevideo,Group 3,Peru,Romania,1,3,A,NaN,Romania,Peru,1930-07-14,Jul,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1930-07-14
4,1930,Uruguay,Montevideo,Group 1,Argentina,France,1,0,H,NaN,Argentina,France,1930-07-15,Jul,Tuesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1930-07-15


In [51]:
silver_wc_clean = silver_wc_clean[silver_wc_clean["year"].between(1998, 2018)].copy()

In [113]:
silver_wc_clean["home_team"] = silver_wc_clean["home_team"].replace(
    "China PR", "China"
)

In [114]:
silver_wc_clean["home_team"] = silver_wc_clean["home_team"].replace(
    "FR Yugoslavia", "Serbia"
)

In [163]:
silver_wc_clean["away_team"] = silver_wc_clean["away_team"].replace(
    "China PR", "China"
)

In [164]:
silver_wc_clean["away_team"] = silver_wc_clean["away_team"].replace(
    "FR Yugoslavia", "Serbia"
)

In [166]:
silver_wc_clean["winning_team"] = silver_wc_clean["winning_team"].replace(
    "China PR", "China"
)

In [167]:
silver_wc_clean["winning_team"] = silver_wc_clean["winning_team"].replace(
    "FR Yugoslavia", "Serbia"
)

In [168]:
silver_wc_clean["losing_team"] = silver_wc_clean["losing_team"].replace(
    "China PR", "China"
)

In [169]:
silver_wc_clean["losing_team"] = silver_wc_clean["losing_team"].replace(
    "FR Yugoslavia", "Serbia"
)

In [170]:
silver_wc_clean["winning_team"] = silver_wc_clean["winning_team"].replace(
    "Portagul", "Portugal"
)

In [171]:
silver_wc_clean["losing_team"] = silver_wc_clean["losing_team"].replace(
    "Columbia", "Colombia"
)

In [172]:
silver_wc_clean

,year,country,city,stage,home_team,away_team,home_score,away_score,outcome,win_conditions,...,losing_team,date,month,dayofweek,source_name,load_timestamp,run_id,load_timestamp_clean,date_clean,review_reason
516,1998,France,Saint-Denis,Group A,Brazil,Brazil,2,1,H,NaN,...,Scotland,1998-06-10,Jun,Wednesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-10,<NA>
517,1998,France,Montpellier,Group A,Morocco,Morocco,2,2,D,NaN,...,NaN,1998-06-10,Jun,Wednesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-10,<NA>
518,1998,France,Toulouse,Group B,Cameroon,Cameroon,1,1,D,NaN,...,NaN,1998-06-11,Jun,Thursday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-11,<NA>
519,1998,France,Bordeaux,Group B,Italy,Italy,2,2,D,NaN,...,NaN,1998-06-11,Jun,Thursday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-11,<NA>
520,1998,France,Marseille,Group C,France,France,3,0,H,NaN,...,South Africa,1998-06-12,Jun,Friday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-12,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,2018,Russia,Sochi,Quarterfinals,Russia,Russia,2,2,A,Croatia won in penalties (4 - 3),...,Russia,2018-07-07,Jul,Saturday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-07-07,<NA>
896,2018,Russia,Saint Petersburg,Semifinals,France,France,1,0,H,NaN,...,Belgium,2018-07-10,Jul,Tuesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-07-10,<NA>
897,2018,Russia,Moscow,Semifinals,Croatia,Croatia,2,1,H,Croatia won in AET,...,England,2018-07-11,Jul,Wednesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-07-11,<NA>
898,2018,Russia,Saint Petersburg,Third place,Belgium,Belgium,2,0,H,NaN,...,England,2018-07-14,Jul,Saturday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-07-14,<NA>


In [173]:
silver_wc_clean[
    silver_wc_clean["winning_team"].isna() &
    silver_wc_clean["losing_team"].isna()
]

,year,country,city,stage,home_team,away_team,home_score,away_score,outcome,win_conditions,...,losing_team,date,month,dayofweek,source_name,load_timestamp,run_id,load_timestamp_clean,date_clean,review_reason
517,1998,France,Montpellier,Group A,Morocco,Morocco,2,2,D,NaN,...,NaN,1998-06-10,Jun,Wednesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-10,<NA>
518,1998,France,Toulouse,Group B,Cameroon,Cameroon,1,1,D,NaN,...,NaN,1998-06-11,Jun,Thursday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-11,<NA>
519,1998,France,Bordeaux,Group B,Italy,Italy,2,2,D,NaN,...,NaN,1998-06-11,Jun,Thursday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-11,<NA>
521,1998,France,Montpellier,Group D,Paraguay,Paraguay,0,0,D,NaN,...,NaN,1998-06-12,Jun,Friday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-12,<NA>
524,1998,France,Saint-Denis,Group E,Netherlands,Netherlands,0,0,D,NaN,...,NaN,1998-06-13,Jun,Saturday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-13,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,2018,Russia,Ekaterinburg,Group H,Japan,Japan,2,2,D,NaN,...,NaN,2018-06-24,Jun,Sunday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-06-24,<NA>
870,2018,Russia,Kaliningrad,Group B,Spain,Spain,2,2,D,NaN,...,NaN,2018-06-25,Jun,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-06-25,<NA>
871,2018,Russia,Saransk,Group B,Iran,Iran,1,1,D,NaN,...,NaN,2018-06-25,Jun,Monday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-06-25,<NA>
873,2018,Russia,Moscow,Group C,Denmark,Denmark,0,0,D,NaN,...,NaN,2018-06-26,Jun,Tuesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,2018-06-26,<NA>


In [174]:
silver_wc_clean["stage"].unique()

<StringArray>
[      'Group A',       'Group B',       'Group C',       'Group D',
       'Group E',       'Group H',       'Group F',       'Group G',
   'Round of 16', 'Quarterfinals',    'Semifinals',   'Third place',
         'Final',      'Group D ']
Length: 14, dtype: str

In [175]:
silver_wc_clean.dtypes

year                             int32
country                            str
city                               str
stage                              str
home_team                          str
away_team                          str
home_score                       int64
away_score                       int64
outcome                            str
win_conditions                     str
winning_team                       str
losing_team                        str
date                               str
month                              str
dayofweek                          str
source_name                        str
load_timestamp                     str
run_id                             str
load_timestamp_clean    datetime64[us]
date_clean              datetime64[us]
review_reason                   object
dtype: object

In [176]:
silver_wc_clean["review_reason"] = pd.NA

silver_wc_clean.loc[
    silver_wc_clean["year"].isna(),"review_reason"] = "MISSING_YEAR"

silver_wc_clean.loc[
    silver_wc_clean["date_clean"].isna(),"review_reason"] = "MISSING_DATE"



In [177]:
silver_review_rows = silver_wc_clean[silver_wc_clean["review_reason"].notna()].copy()

In [178]:
silver_review_rows

,year,country,city,stage,home_team,away_team,home_score,away_score,outcome,win_conditions,...,losing_team,date,month,dayofweek,source_name,load_timestamp,run_id,load_timestamp_clean,date_clean,review_reason


In [179]:
silver_wc_clean.columns

Index(['year', 'country', 'city', 'stage', 'home_team', 'away_team',
       'home_score', 'away_score', 'outcome', 'win_conditions', 'winning_team',
       'losing_team', 'date', 'month', 'dayofweek', 'source_name',
       'load_timestamp', 'run_id', 'load_timestamp_clean', 'date_clean',
       'review_reason'],
      dtype='string')

In [180]:
silver_wc_clean.head(2)

,year,country,city,stage,home_team,away_team,home_score,away_score,outcome,win_conditions,...,losing_team,date,month,dayofweek,source_name,load_timestamp,run_id,load_timestamp_clean,date_clean,review_reason
516,1998,France,Saint-Denis,Group A,Brazil,Brazil,2,1,H,NaN,...,Scotland,1998-06-10,Jun,Wednesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-10,<NA>
517,1998,France,Montpellier,Group A,Morocco,Morocco,2,2,D,NaN,...,NaN,1998-06-10,Jun,Wednesday,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe,2026-06-29 17:08:30,1998-06-10,<NA>


In [181]:
silver_wc_output = silver_wc_clean[["year", "country", "city", "stage", "home_team", "away_team", "home_score", "away_score", "winning_team", "losing_team", "date_clean", "source_name", "load_timestamp_clean", "run_id" ]].copy()

In [182]:
silver_wc_output.tail()

,year,country,city,stage,home_team,away_team,home_score,away_score,winning_team,losing_team,date_clean,source_name,load_timestamp_clean,run_id
895,2018,Russia,Sochi,Quarterfinals,Russia,Russia,2,2,Croatia,Russia,2018-07-07,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
896,2018,Russia,Saint Petersburg,Semifinals,France,France,1,0,France,Belgium,2018-07-10,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
897,2018,Russia,Moscow,Semifinals,Croatia,Croatia,2,1,Croatia,England,2018-07-11,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
898,2018,Russia,Saint Petersburg,Third place,Belgium,Belgium,2,0,Belgium,England,2018-07-14,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe
899,2018,Russia,Moscow,Final,France,France,4,2,France,Croatia,2018-07-15,fifa_world_cup_archive,2026-06-29 17:08:30,46875d21-433c-4d67-be89-7b8b640985fe


In [183]:
silver_wc_output.to_csv(output_path / "silver_wc_output.csv", index=False )

In [184]:
silver_wc_summary = pd.DataFrame([
    {"metric_name": "bronze_wc_rows", "metric_value": len(pd.read_csv(bronze_dir / "bronze_wc_matches.csv")), "explanation": "original, all years"},
    {"metric_name": "silver_wc_rows", "metric_value": len(pd.read_csv(silver_dir / "silver_wc_output.csv")), "explanation": "filtered 1998-2018"},
    {"metric_name": "rejected_row", "metric_value": len(silver_review_rows), "explanation": "nothing rejected"}
])

In [185]:
silver_wc_summary

,metric_name,metric_value,explanation
0,bronze_wc_rows,900,"original, all years"
1,silver_wc_rows,384,filtered 1998-2018
2,rejected_row,0,nothing rejected


In [186]:
with open(silver_dir / "silver_wc_summary.txt", "w") as f:
    f.write(silver_wc_summary.to_string(index=False))